In [1]:
import yfinance as yf
import pandas as pd

stock = yf.Ticker("AAPL")
df = stock.history(period="5y")
df.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2021-05-11 00:00:00-04:00,120.492766,123.195313,119.780538,122.844086,126142800,0.0,0.0
2021-05-12 00:00:00-04:00,120.395205,121.605009,119.273206,119.780540,112172300,0.0,0.0
2021-05-13 00:00:00-04:00,121.546476,123.078246,121.234268,121.926979,105861300,0.0,0.0
2021-05-14 00:00:00-04:00,123.175811,124.775877,122.785550,124.346588,81918000,0.0,0.0
2021-05-17 00:00:00-04:00,123.731915,123.839237,122.122092,123.195305,74244600,0.0,0.0


In [2]:
info = stock.info

print(f"Company: {info.get('longName')}")
print(f"Sector: {info.get('sector')}")
print(f"Industry: {info.get('industry')}")
print(f"Market Cap: ${info.get('marketCap', 0):,.0f}")
print(f"P/E Ratio: {info.get('trailingPE', 'N/A')}")
print(f"Revenue Growth: {info.get('revenueGrowth', 'N/A')}")
print(f"Profit Margin: {info.get('profitMargins', 'N/A')}")
print(f"Debt to Equity: {info.get('debtToEquity', 'N/A')}")
print(f"\nTotal rows of price data: {df.shape[0]}")

Company: Apple Inc.
Sector: Technology
Industry: Consumer Electronics
Market Cap: $4,278,353,657,856
P/E Ratio: 35.223095
Revenue Growth: 0.166
Profit Margin: 0.27152002
Debt to Equity: 79.548

Total rows of price data: 1256


In [3]:
def calculate_rsi(data, period=14):
    delta = data['Close'].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df['RSI'] = calculate_rsi(df)
df[['Close', 'RSI']].tail(10)

,Close,RSI
Date,,
2026-04-28 00:00:00-04:00,270.709991,62.674400
2026-04-29 00:00:00-04:00,270.170013,60.628057
2026-04-30 00:00:00-04:00,271.350006,61.635629
2026-05-01 00:00:00-04:00,280.140015,69.310230
2026-05-04 00:00:00-04:00,276.829987,65.745284
2026-05-05 00:00:00-04:00,284.179993,65.594806
2026-05-06 00:00:00-04:00,287.510010,71.071508
2026-05-07 00:00:00-04:00,287.440002,67.056489
2026-05-08 00:00:00-04:00,293.320007,68.940400


In [4]:
# MACD - Moving Average Convergence Divergence
# Measures momentum: when short-term trend crosses long-term trend
def calculate_macd(data):
    ema12 = data['Close'].ewm(span=12).mean()
    ema26 = data['Close'].ewm(span=26).mean()
    macd_line = ema12 - ema26
    signal_line = macd_line.ewm(span=9).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

df['MACD'], df['MACD_Signal'], df['MACD_Hist'] = calculate_macd(df)

# Bollinger Bands - Shows if price is unusually high or low
# Price above upper band = potentially overbought
# Price below lower band = potentially oversold
def calculate_bollinger(data, period=20):
    sma = data['Close'].rolling(window=period).mean()
    std = data['Close'].rolling(window=period).std()
    upper = sma + (2 * std)
    lower = sma - (2 * std)
    # Where is price relative to the bands? 0=lower band, 1=upper band
    bb_position = (data['Close'] - lower) / (upper - lower)
    return upper, lower, bb_position

df['BB_Upper'], df['BB_Lower'], df['BB_Position'] = calculate_bollinger(df)

# Moving averages - trend direction
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['SMA_200'] = df['Close'].rolling(window=200).mean()

# Volume ratio - is today's volume unusual?
df['Volume_Ratio'] = df['Volume'] / df['Volume'].rolling(window=20).mean()

# Volatility - how much does price swing?
df['Volatility'] = df['Close'].pct_change().rolling(window=30).std()

# Daily return
df['Daily_Return'] = df['Close'].pct_change()

print("All indicators calculated!")
print(f"\nColumns now: {df.columns.tolist()}")
df[['Close', 'RSI', 'MACD', 'BB_Position', 'SMA_50', 'SMA_200', 'Volume_Ratio', 'Volatility']].tail(5)

All indicators calculated!

Columns now: ['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper', 'BB_Lower', 'BB_Position', 'SMA_50', 'SMA_200', 'Volume_Ratio', 'Volatility', 'Daily_Return']


,Close,RSI,MACD,BB_Position,SMA_50,SMA_200,Volume_Ratio,Volatility
Date,,,,,,,,
2026-05-05 00:00:00-04:00,284.179993,65.594806,5.248893,1.036123,261.823600,255.606807,1.071286,0.015744
2026-05-06 00:00:00-04:00,287.510010,71.071508,5.994972,1.046127,262.131000,255.985181,1.243959,0.015787
2026-05-07 00:00:00-04:00,287.440002,67.056489,6.505603,0.969757,262.395200,256.353633,0.947093,0.015811
2026-05-08 00:00:00-04:00,293.320007,68.940400,7.300592,1.039195,262.802599,256.752732,1.078118,0.016063
2026-05-11 00:00:00-04:00,291.239990,75.733884,7.674322,0.930496,263.343799,257.143374,0.507309,0.015731


In [5]:
# THE PREDICTION TARGET
# "Did the stock go UP or DOWN over the next 30 days?"
# 1 = went up, 0 = went down

df['Future_Return'] = df['Close'].shift(-30) / df['Close'] - 1
df['Target'] = (df['Future_Return'] > 0).astype(int)

print(f"Total rows: {df.shape[0]}")
print(f"Rows with target: {df['Target'].dropna().shape[0]}")
print(f"\nTarget distribution:")
print(df['Target'].value_counts())
print(f"\n% that went up: {df['Target'].mean()*100:.1f}%")

Total rows: 1256
Rows with target: 1256

Target distribution:
Target
1    724
0    532
Name: count, dtype: int64

% that went up: 57.6%


In [6]:
# ALL YOUR STOCKS BY SECTOR
stocks = {
    'Tech': ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 
             'RDDT', 'AMD', 'CRM', 'INTC', 'NFLX', 'SHOP', 'UBER', 'SNAP'],
    'Finance': ['JPM', 'GS', 'V', 'MA', 'PYPL', 'SQ'],
    'Real Estate': ['O', 'AMT', 'PLD', 'SPG', 'WELL', 'DLR'],
    'Defence': ['LMT', 'RTX', 'NOC', 'GD', 'BA']
}

all_data = []

for sector, tickers in stocks.items():
    for ticker in tickers:
        try:
            print(f"Pulling {ticker}...", end=" ")
            s = yf.Ticker(ticker)
            d = s.history(period="5y")
            
            if len(d) < 200:
                print("SKIPPED (not enough data)")
                continue
            
            # Calculate all indicators
            d['RSI'] = calculate_rsi(d)
            d['MACD'], d['MACD_Signal'], d['MACD_Hist'] = calculate_macd(d)
            d['BB_Upper'], d['BB_Lower'], d['BB_Position'] = calculate_bollinger(d)
            d['SMA_50'] = d['Close'].rolling(window=50).mean()
            d['SMA_200'] = d['Close'].rolling(window=200).mean()
            d['Volume_Ratio'] = d['Volume'] / d['Volume'].rolling(window=20).mean()
            d['Volatility'] = d['Close'].pct_change().rolling(window=30).std()
            d['Daily_Return'] = d['Close'].pct_change()
            
            # Prediction target
            d['Future_Return'] = d['Close'].shift(-30) / d['Close'] - 1
            d['Target'] = (d['Future_Return'] > 0).astype(int)
            
            # Get fundamentals
            info = s.info
            d['Ticker'] = ticker
            d['Sector'] = sector
            d['Market_Cap'] = info.get('marketCap', 0)
            d['PE_Ratio'] = info.get('trailingPE', 0)
            d['Revenue_Growth'] = info.get('revenueGrowth', 0)
            d['Profit_Margin'] = info.get('profitMargins', 0)
            
            # Classify market cap
            mc = info.get('marketCap', 0)
            if mc > 200e9:
                d['Cap_Size'] = 'Large'
            elif mc > 10e9:
                d['Cap_Size'] = 'Mid'
            else:
                d['Cap_Size'] = 'Small'
            
            all_data.append(d)
            print(f"OK ({len(d)} rows)")
            
        except Exception as e:
            print(f"ERROR: {e}")

# Combine everything
master_df = pd.concat(all_data)
print(f"\n{'='*50}")
print(f"Total rows: {master_df.shape[0]}")
print(f"Total stocks: {master_df['Ticker'].nunique()}")
print(f"Sectors: {master_df['Sector'].unique().tolist()}")
print(f"\nRows per sector:")
print(master_df.groupby('Sector')['Ticker'].count())


Pulling AAPL... OK (1256 rows)
Pulling MSFT... OK (1256 rows)
Pulling GOOGL... OK (1256 rows)
Pulling AMZN... OK (1256 rows)
Pulling NVDA... OK (1256 rows)
Pulling META... OK (1256 rows)
Pulling TSLA... OK (1256 rows)
Pulling RDDT... OK (536 rows)
Pulling AMD... OK (1256 rows)
Pulling CRM... OK (1256 rows)
Pulling INTC... OK (1256 rows)
Pulling NFLX... OK (1256 rows)
Pulling SHOP... OK (1256 rows)
Pulling UBER... OK (1256 rows)
Pulling SNAP... OK (1256 rows)
Pulling JPM... OK (1256 rows)
Pulling GS... OK (1256 rows)
Pulling V... OK (1256 rows)
Pulling MA... OK (1256 rows)
Pulling PYPL... OK (1256 rows)
Pulling SQ... 

$SQ: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")


SKIPPED (not enough data)
Pulling O... OK (1256 rows)
Pulling AMT... OK (1256 rows)
Pulling PLD... OK (1256 rows)
Pulling SPG... OK (1256 rows)
Pulling WELL... OK (1256 rows)
Pulling DLR... OK (1256 rows)
Pulling LMT... OK (1256 rows)
Pulling RTX... OK (1256 rows)
Pulling NOC... OK (1256 rows)
Pulling GD... OK (1256 rows)
Pulling BA... OK (1256 rows)

Total rows: 38216
Total stocks: 31
Sectors: ['Tech', 'Finance', 'Real Estate', 'Defence']

Rows per sector:
Sector
Defence         6280
Finance         6280
Real Estate     7536
Tech           18120
Name: Ticker, dtype: int64


In [7]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# Features your model will learn from
feature_cols = ['RSI', 'MACD', 'MACD_Hist', 'BB_Position', 
                'Volume_Ratio', 'Volatility', 'Daily_Return',
                'PE_Ratio', 'Revenue_Growth', 'Profit_Margin']

# Drop rows with missing values (first 200 days have NaN from rolling calculations)
model_df = master_df.dropna(subset=feature_cols + ['Target'])

print(f"Rows after cleaning: {model_df.shape[0]}")
print(f"Target distribution:")
print(model_df['Target'].value_counts())

X = model_df[feature_cols]
y = model_df['Target']

# Time-based split — NOT random
# Use first 80% for training, last 20% for testing
# This prevents look-ahead bias (crucial for financial data)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"\nTraining set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")

# Train AdaBoost
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3),
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

ada.fit(X_train, y_train)

# Evaluate
y_pred = ada.predict(X_test)
print(f"\n{'='*50}")
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.1f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['DOWN', 'UP']))

Rows after cleaning: 37286
Target distribution:
Target
1    20253
0    17033
Name: count, dtype: int64

Training set: 29828 rows
Test set: 7458 rows

Accuracy: 53.3%

Classification Report:
              precision    recall  f1-score   support

        DOWN       0.43      0.23      0.30      3251
          UP       0.56      0.77      0.65      4207

    accuracy                           0.53      7458
   macro avg       0.50      0.50      0.47      7458
weighted avg       0.51      0.53      0.50      7458



In [10]:
import plotly.express as px

importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': ada.feature_importances_
}).sort_values('Importance', ascending=True)

fig = px.bar(importance, x='Importance', y='Feature', orientation='h',
             title='AdaBoost Feature Importance',
             color='Importance', color_continuous_scale='teal')
fig.update_layout(height=400, showlegend=False)
fig.show()

print("\nFeature importance ranking:")
for _, row in importance.sort_values('Importance', ascending=False).iterrows():
    bar = '█' * int(row['Importance'] * 100)
    print(f"  {row['Feature']:20s} {row['Importance']*100:.1f}%  {bar}")


Feature importance ranking:
  Volatility           36.2%  ████████████████████████████████████
  MACD_Hist            21.0%  ████████████████████
  MACD                 14.1%  ██████████████
  PE_Ratio             12.8%  ████████████
  Revenue_Growth       8.1%  ████████
  Profit_Margin        3.9%  ███
  BB_Position          1.6%  █
  RSI                  1.6%  █
  Volume_Ratio         0.5%  
  Daily_Return         0.2%  


In [11]:
test_df = model_df.iloc[split_idx:].copy()
test_df['Predicted'] = y_pred

print("Accuracy by sector:")
print("="*40)
for sector in test_df['Sector'].unique():
    sector_data = test_df[test_df['Sector'] == sector]
    acc = accuracy_score(sector_data['Target'], sector_data['Predicted'])
    print(f"  {sector:15s} → {acc*100:.1f}%  ({len(sector_data)} rows)")

Accuracy by sector:
  Real Estate     → 56.2%  (1328 rows)
  Defence         → 52.6%  (6130 rows)


In [12]:
from groq import Groq

# Replace with your actual key
client = Groq(api_key="REDACTED_USE_ENV_GROQ_API_KEY")

def analyse_stock(ticker):
    # Pull fresh data
    s = yf.Ticker(ticker)
    d = s.history(period="5y")
    info = s.info
    
    # Calculate indicators
    d['RSI'] = calculate_rsi(d)
    d['MACD'], d['MACD_Signal'], d['MACD_Hist'] = calculate_macd(d)
    d['BB_Upper'], d['BB_Lower'], d['BB_Position'] = calculate_bollinger(d)
    d['SMA_50'] = d['Close'].rolling(window=50).mean()
    d['SMA_200'] = d['Close'].rolling(window=200).mean()
    d['Volume_Ratio'] = d['Volume'] / d['Volume'].rolling(window=20).mean()
    d['Volatility'] = d['Close'].pct_change().rolling(window=30).std()
    d['Daily_Return'] = d['Close'].pct_change()
    
    # Get latest values
    latest = d.iloc[-1]
    
    # Model prediction
    features = pd.DataFrame([[
        latest['RSI'], latest['MACD'], latest['MACD_Hist'],
        latest['BB_Position'], latest['Volume_Ratio'], latest['Volatility'],
        latest['Daily_Return'], info.get('trailingPE', 0),
        info.get('revenueGrowth', 0), info.get('profitMargins', 0)
    ]], columns=feature_cols)
    
    prediction = ada.predict(features)[0]
    probability = ada.predict_proba(features)[0]
    confidence = max(probability) * 100
    direction = "UP" if prediction == 1 else "DOWN"
    
    # Feature importance ranking
    importance_text = "\n".join([
        f"  {i+1}. {feat}: {imp*100:.1f}% importance — current value: {features[feat].values[0]:.4f}"
        for i, (feat, imp) in enumerate(
            sorted(zip(feature_cols, ada.feature_importances_), 
                   key=lambda x: x[1], reverse=True)
        )
    ])
    
    # Build prompt for Groq
    prompt = f"""You are a financial analyst AI. Analyse this stock based on the ML model's output.

STOCK: {ticker} ({info.get('longName', ticker)})
SECTOR: {info.get('sector', 'Unknown')}
MARKET CAP: ${info.get('marketCap', 0):,.0f}
CURRENT PRICE: ${latest['Close']:.2f}

ML MODEL PREDICTION: {direction} in next 30 days
CONFIDENCE: {confidence:.1f}%

CURRENT INDICATORS (ranked by model importance):
{importance_text}

FUNDAMENTALS:
  P/E Ratio: {info.get('trailingPE', 'N/A')}
  Revenue Growth: {info.get('revenueGrowth', 'N/A')}
  Profit Margin: {info.get('profitMargins', 'N/A')}
  Debt to Equity: {info.get('debtToEquity', 'N/A')}

SMA 50: ${latest.get('SMA_50', 0):.2f}
SMA 200: ${latest.get('SMA_200', 0):.2f}

RULES:
1. Structure your analysis by importance ranking — most important factor first
2. Cite every number from the data above
3. Explain what each indicator means in plain English
4. Be honest about model confidence — {confidence:.1f}% is {'high' if confidence > 60 else 'moderate' if confidence > 55 else 'low'}
5. NEVER give buy/sell advice — only analysis
6. End with key risks to watch
7. Keep it under 300 words"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=500
    )
    
    return {
        'ticker': ticker,
        'company': info.get('longName', ticker),
        'sector': info.get('sector', 'Unknown'),
        'price': latest['Close'],
        'direction': direction,
        'confidence': confidence,
        'analysis': response.choices[0].message.content,
        'indicators': {feat: latest.get(feat, 0) for feat in ['RSI', 'MACD', 'BB_Position', 'Volatility', 'Volume_Ratio']},
        'importance': dict(zip(feature_cols, ada.feature_importances_))
    }

# TEST IT
print("Analysing AAPL...")
result = analyse_stock("AAPL")
print(f"\n{'='*60}")
print(f"Stock: {result['company']} ({result['ticker']})")
print(f"Price: ${result['price']:.2f}")
print(f"Prediction: {result['direction']} ({result['confidence']:.1f}% confident)")
print(f"\n{'='*60}")
print(f"AI ANALYSIS:\n")
print(result['analysis'])

Analysing AAPL...

Stock: Apple Inc. (AAPL)
Price: $291.58
Prediction: UP (52.5% confident)

AI ANALYSIS:

Analyzing AAPL, the most important factor is Volatility (36.2% importance), currently at 0.0157. This measures the stock's price fluctuations, indicating a relatively stable price. The MACD_Hist (21.0% importance) is 1.9118, showing the difference between the stock's price and its moving averages, suggesting a potential upward trend. The MACD (14.1% importance) is 7.7012, indicating the stock's momentum.

The PE_Ratio (12.8% importance) is 35.2572, meaning investors are willing to pay $35.26 for every dollar of earnings. Revenue_Growth (8.1% importance) is 0.1660, indicating a moderate growth rate. Profit_Margin (3.9% importance) is 0.2715, showing the company's ability to maintain profitability.

With a model confidence of 52.5%, the prediction of an upward trend in the next 30 days is not strongly supported. The current price ($291.58) is above the SMA 50 ($263.35) and SMA 200 (

In [13]:
# Test news fetching for a stock
import requests

def get_stock_news(ticker):
    """Get recent news using Yahoo Finance RSS"""
    s = yf.Ticker(ticker)
    try:
        news = s.news
        headlines = []
        for item in news[:10]:
            headlines.append({
                'title': item.get('title', ''),
                'publisher': item.get('publisher', ''),
                'link': item.get('link', ''),
            })
        return headlines
    except:
        return []

# Test it
news = get_stock_news("AAPL")
print(f"Found {len(news)} news articles:\n")
for i, n in enumerate(news):
    print(f"  {i+1}. [{n['publisher']}] {n['title']}")

Found 10 news articles:

  1. [] 
  2. [] 
  3. [] 
  4. [] 
  5. [] 
  6. [] 
  7. [] 
  8. [] 
  9. [] 
  10. [] 


In [14]:
# Let's see what the news data actually looks like
s = yf.Ticker("AAPL")
news = s.news

# Print raw structure to understand it
for item in news[:3]:
    print(item)
    print(type(item))
    print("Keys:", item.keys() if isinstance(item, dict) else "not a dict")
    print("---")

{'id': '264f0653-66c1-4269-91f9-9988f980b3fe', 'content': {'id': '264f0653-66c1-4269-91f9-9988f980b3fe', 'contentType': 'VIDEO', 'title': 'Memory chip stocks hit record highs, pharma reacts to hantavirus concerns', 'description': '<p>Market Catalysts Host Julie Hyman and Yahoo Finance Senior Reporter Brooke DiPalma track several of the day\'s top trending stock tickers, including memory chip stocks Micron Technology (<a target="_blank" rel="" class="link" href="https://finance.yahoo.com/quote/MU/" data-i13n="cpos:1;pos:1">MU</a>), Intel (<a target="_blank" rel="" class="link" href="https://finance.yahoo.com/quote/INTC/" data-i13n="cpos:2;pos:1">INTC</a>), and Qualcomm (<a target="_blank" rel="" class="link" href="https://finance.yahoo.com/quote/QCOM/" data-i13n="cpos:3;pos:1">QCOM</a>) opening at record highs; shares of Moderna (<a target="_blank" rel="" class="link" href="https://finance.yahoo.com/quote/MRNA/" data-i13n="cpos:4;pos:1">MRNA</a>) and Novavax (<a target="_blank" rel="" c

In [15]:
def get_stock_news(ticker):
    s = yf.Ticker(ticker)
    try:
        news = s.news
        headlines = []
        for item in news[:10]:
            content = item.get('content', {})
            headlines.append({
                'title': content.get('title', ''),
                'publisher': content.get('provider', {}).get('displayName', ''),
                'summary': content.get('summary', ''),
                'date': content.get('pubDate', ''),
                'link': content.get('canonicalUrl', {}).get('url', ''),
            })
        return headlines
    except:
        return []

# Test it
news = get_stock_news("AAPL")
print(f"Found {len(news)} articles:\n")
for i, n in enumerate(news):
    print(f"  {i+1}. [{n['publisher']}] {n['title']}")
    print(f"     {n['summary'][:100]}...")
    print()
    

Found 10 articles:

  1. [Yahoo Finance Video] Memory chip stocks hit record highs, pharma reacts to hantavirus concerns
     Market Catalysts Host Julie Hyman and Yahoo Finance Senior Reporter Brooke DiPalma track several of ...

  2. [Yahoo Finance] The $1 trillion club's new members are powering the AI boom: Chart of the Day
     Market royalty is getting a hardware makeover....

  3. [Motley Fool] Why One Fund Made a $23 Million Bet on This Addiction-Treatment Stock Amid a Staggering Rally
     Indivior develops proprietary treatments for opioid dependence, generating revenue from specialty ph...

  4. [Bloomberg] Musk, Cook Set to Join Trump for Xi Summit, White House Says
     (Bloomberg) -- The White House is inviting Tesla Inc.’s Elon Musk, Apple Inc.’s Tim Cook, Boeing Co....

  5. [The Wall Street Journal] Who’s Traveling to China With Trump in the U.S. Business Delegation?
     More than a dozen top U.S. business leaders will travel to China for President Trump’s visit this 

In [16]:
import pickle

with open('ada_model.pkl', 'wb') as f:
    pickle.dump(ada, f)

with open('model_config.pkl', 'wb') as f:
    pickle.dump({
        'feature_cols': feature_cols,
        'importance': dict(zip(feature_cols, ada.feature_importances_))
    }, f)

print("Model saved!")

Model saved!
